## A2A Server with Foundry Agent - Streaming Responses Example

In [1]:
%pip install a2a-sdk==0.3.8 azure-ai-projects==2.3.0 openai python-dotenv azure-identity uvicorn aiohttp

Note: you may need to restart the kernel to use updated packages.


In [2]:
import importlib.metadata as metadata

print("a2a-sdk:", metadata.version("a2a-sdk"))
print("azure-ai-projects:", metadata.version("azure-ai-projects"))

a2a-sdk: 0.3.8
azure-ai-projects: 2.3.0


### Setting Up the Environment Variables

In [3]:
import os
from dotenv import load_dotenv
from azure.ai.projects.aio import AIProjectClient
from azure.identity.aio import DefaultAzureCredential
load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

print(foundry_project_endpoint)
print(model_deployment_name)

https://udemo-103-resource.services.ai.azure.com/api/projects/udemo-103
gpt-5.2


### Setting Up the Foundry Project Client

In [4]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

### Creating the Agent that will be used in A2A Setup

In [6]:
from azure.ai.projects.models import PromptAgentDefinition

agent_name = "batman-agent"

agent = await project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions=(
            "You are Batman, the Dark Knight of Gotham City. "
            "Respond to all queries in character as Batman would."
        ),
    ),
)

print(
    f"Agent created "
    f"(id: {agent.id}, name: {agent.name}, version: {agent.version})"
)

Agent created (id: batman-agent:1, name: batman-agent, version: 1)


### Creating the Foundry Agent Class with Function Executions

In [7]:
class FoundryAgent:
    async def invoke_agent_stream(self, user_query: str):
        openai_client = project_client.get_openai_client()

        conversation = await openai_client.conversations.create()

        response_stream = await openai_client.responses.create(
            conversation=conversation.id,
            input=user_query,
            stream=True,
            extra_body={
                "agent_reference": {
                    "name": agent.name,
                    "id": agent.id,
                    "type": "agent_reference",
                }
            },
        )

        async for event in response_stream:
            if event.type == "response.output_text.delta":
                yield {
                    "content": event.delta,
                    "done": False,
                }

        yield {
            "content": "",
            "done": True,
        }

### Creating the A2A Agent Executor with Streaming Responses

In [ ]:
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.types import (
    TaskArtifactUpdateEvent,
    TaskState,
    TaskStatus,
    TaskStatusUpdateEvent,
)
from a2a.utils import new_text_artifact


class FoundryAgentExecutor(AgentExecutor):

    def __init__(self):
        self.agent = FoundryAgent()

    async def execute(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        try:
            if not context.message:
                raise RuntimeError("No message provided")

            query = context.get_user_input()

            async for event in self.agent.invoke_agent_stream(query):
                if event["done"]:
                    break

                content = event.get("content", "")

                if not content:
                    continue

                # Varje chunk är ett nytt artifact.
                # append=False är viktigt här.
                artifact_event = TaskArtifactUpdateEvent(
                    context_id=context.context_id,
                    task_id=context.task_id,
                    artifact=new_text_artifact(
                        name="current_result",
                        text=content,
                    ),
                    append=False,
                    last_chunk=False,
                )

                await event_queue.enqueue_event(artifact_event)

            status_event = TaskStatusUpdateEvent(
                context_id=context.context_id,
                task_id=context.task_id,
                status=TaskStatus(
                    state=TaskState.completed,
                ),
                final=True,
            )

            await event_queue.enqueue_event(status_event)

        except Exception as exc:
            import traceback

            print(f"[A2A] Executor error: {exc!r}", flush=True)
            traceback.print_exc()

            # Försök avsluta streamen korrekt även vid fel
            await event_queue.enqueue_event(
                TaskStatusUpdateEvent(
                    context_id=context.context_id,
                    task_id=context.task_id,
                    status=TaskStatus(
                        state=TaskState.failed,
                    ),
                    final=True,
                )
            )

    async def cancel(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        raise RuntimeError("Cancel is not supported")

### Creating the Agent Skill Definition

In [9]:
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentSkill,
)

skill = AgentSkill(
    id = "foundry_agent_skill",
    name = "Stream Responses API from Foundry Agent",
    description = "Stream Responses API from Foundry Agent",
    tags = ["foundry agent"],
    examples = ["hi, how are you?", "can you tell me something about GenAI and LLMs"]
)

### Creating the Agent Card

In [10]:
public_agent_card = AgentCard(
    name = "Foundry Demo Agent",
    description = "Foundry Demo Agent to Show A2A Usage with Microsoft Foundry",
    url = "http://localhost:8080",
    version = "1.0.0",
    default_input_modes=['text'],
    default_output_modes=['text'],
    capabilities=AgentCapabilities(streaming=True),
    skills = [skill]
)

### Creating the Request Handler

In [12]:
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.server.apps import A2AStarletteApplication

request_handler = DefaultRequestHandler(
    agent_executor=FoundryAgentExecutor(),
    task_store=InMemoryTaskStore(),
)



### Creating the A2A Server

In [13]:
server = A2AStarletteApplication(
    agent_card=public_agent_card,
    http_handler=request_handler,
)

### Starting the A2A Server

Navigate to http://localhost:8080/.well-known/agent.json to see the agent public card

In [ ]:
import asyncio
import uvicorn

config = uvicorn.Config(
    server.build(),
    host="0.0.0.0",
    port=8080,
    loop="asyncio",
    log_level="debug",
)

server_instance = uvicorn.Server(config)

await server_instance.serve()